In [ ]:
import sys
sys.path.append('..')

import torch
from source.evaluation.model_loading import load_model_and_tokenizer
from source.evaluation.evaluation import evaluate_model
from source.data_preprocessing import load_and_preprocess_multiwoz

torch.cuda.empty_cache()
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

In [ ]:

# Evaluation parameters
EVAL_PARAMS = {
    "max_seq_len": 256,
    "eval_batch_size": 8,
    "gen_max_new_tokens": 256,
    "repetition_penalty": 1.35
}

In [ ]:
# Define model paths
# Pre‑trained (output of training comparison)
pretrained_paths = {
    "Baseline DeepSeekMoE": "./checkpoints/baseline/final",
    "DYNMoE baseline": "./checkpoints/dynmoe_baseline/final",
    "Prototype (DeepSeekMoE + DYNMoE routing)": "./checkpoints/dynmoe_routing/final",
}

# Fine‑tuned (output of fine‑tuning comparison)
finetuned_paths = {
    "Baseline DeepSeekMoE (fine‑tuned)": "./checkpoints/baseline-ft/final",
    "DYNMoE baseline (fine‑tuned)": "./checkpoints/dynmoe_baseline-ft/final",
    "Prototype (fine‑tuned)": "./checkpoints/dynmoe_routing-ft/final",
}

In [ ]:
train_sequences, val_sequences, test_sequences = load_and_preprocess_multiwoz(
    zip_path="MultiWOZ-coref/MultiWOZ2_3.zip",
    sample_size=300,
    random_seed=42
)

print(f"Test sequences available: {len(test_sequences)}")

In [ ]:
# Evaluation function 
def run_evaluation(label, model_path):
    print("=" * 60)
    print(f"EVALUATING: {label}")
    print(f"Model path: {model_path}")
    print("=" * 60)

    model, tokenizer = load_model_and_tokenizer(model_path)
    model = model.to(device)
    model.eval()
    print(f"Model loaded from {model_path}")

    results = evaluate_model(
        model,
        tokenizer,
        "test_sequences.txt",
        device,
        **EVAL_PARAMS
    )
    
    print("Evaluation results:")
    for key, value in results.items():
        print(f"  {key}: {value}")
    print()
    return results

In [ ]:
# Run all evaluations
# Pre‑trained models
for label, path in pretrained_paths.items():
    run_evaluation(label, path)

# Fine‑tuned models
for label, path in finetuned_paths.items():
    run_evaluation(label, path)

print("All evaluations completed.")